# Pooled regions 2: motifbreakR vs motifbreakR + CAV over every MPRA region

Meeting 2026-10-27, ask 1. Each single region has only one to three strong peaks, so one wrong motif choice decides
its correlation. Here every MPRA region is evaluated with the same rules, and the comparison is summarised over all of
them. **CPU only**: reads what `Pooled Regions 1 - Sites and Fits for Every MPRA Region.ipynb` wrote to `fcav_pool/`.

The rules are those of the earlier notebooks, copied unchanged:

| rule | motif used at each SNV | prediction |
|---|---|---|
| motifbreakR | largest \|alleleDiff\| | alleleDiff |
| CAV-chosen motifbreakR | highest P(bound) | alleleDiff |
| motifbreakR + CAV (the one shown at the meetings) | highest P(bound) | alleleDiff × w(P), w = sigmoid(20 (P − 0.75)) |

Four scores per region: per-SNV Pearson and Spearman of |·| (the old ones), Pearson after 5 bp smoothing, and F1 of
hits/misses with ±2 bp tolerance (from the tolerant-metric notebook).

**Three groups of regions**: all; the regions measured in **HepG2 or K562** (the cells the CAV was trained on); the
rest (other cell types, where the CAV is used out of its cell type).

Sections:
1. Load, and a check against the earlier per-region fits.
2. Per-region scores, and the pooled comparison with two nulls.
3. Does the gain depend on the number of peaks, or on the CAV's confidence? (21 regions instead of 3.)
4. The per-occurrence calibration of the last notebook (Part C), now with ~10x more occurrences, and with TFs that
   occur in several regions, so the within-motif null finally has something to shuffle.

In [ ]:
!pip install -q scipy pandas matplotlib
from google.colab import drive
drive.mount("/content/drive")

## 1.1 Settings

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

POOL = f"{WORK_DIR}/fcav_pool"
OUT = f"{WORK_DIR}/mpra_eval_pool"
os.makedirs(OUT, exist_ok=True)
MODELS = ["nt500m", "dnabert2", "genalm", "hyenadna", "nt2p5b"]   # those with fits in fcav_pool/fits are used
LAYER_FIX = {}              # model -> layer; default: its only fitted layer, or its best mean AUC over TFs
HOME = {"HepG2", "K562"}    # the cells the CAV was trained on
MIN_SNV = 20                # a region is evaluated when at least this many SNVs get a motifbreakR call
LEGACY = {0: "fcav_pgc_hk_lr_r0", 8: "fcav_pgc_hk_lr", 20: "fcav_pgc_hk_lr_r20"}   # earlier per-region NT-v2 500M fits

MID, K = 0.75, 20           # w(P) = sigmoid(K (P - MID)), as in Hit Selection
MBR_THRESH, MBR_NEUT = 0.85, 0.4
W_MAIN = 5                  # smoothing width (bp)
K_MAIN = 2                  # tolerance (bp)
EFFECT_TOP = 0.15           # "measured effect" = the strongest 15% of bases by |mean log2FC|
PRED_MIN = 0.1              # a base counts as "predicted" when |prediction| >= this
P_CONF = 0.75               # "confident" = P(bound) >= this
PEAK_FRAC = 0.5             # a peak = a run of >= 3 bp where 5 bp-smoothed |log2FC| >= PEAK_FRAC x the region's maximum
N_PERM = 1000

C = json.load(open(f"{WORK_DIR}/config.json"))
MWJ = json.load(open(f"{POOL}/mpra_windows.json"))
INFO = {int(k): v for k, v in MWJ["regions"].items()}
INK = "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": INK,
                     "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK})
pd.set_option("display.width", 250)

## 1.2 Load every region and every model's P(bound)

`build` and `predict` are the code of the tolerant-metric notebook; only the paths change (and the per-base track is
computed with numpy instead of pandas, same numbers, so the 1000 shuffles × 21 regions × 5 models stay fast).

In [ ]:
wfun = lambda x: 1 / (1 + np.exp(-K * (x - MID)))


def _read_meme(path):
    out, name, rows_, ns = {}, None, [], 20
    for line in open(path):
        t = line.split()
        if line.startswith("MOTIF"):
            if name is not None and name not in out:
                out[name] = (np.array(rows_), ns)
            name, rows_, ns = (t[2] if len(t) > 2 else t[1]), [], 20
        elif line.startswith("letter-probability"):
            m = re.search(r"nsites=\s*([0-9.]+)", line)
            ns = float(m.group(1)) if m else 20
        elif name is not None and len(t) == 4 and all(re.fullmatch(r"[0-9.eE+-]+", x) for x in t):
            rows_.append([float(x) for x in t])
    if name is not None and name not in out:
        out[name] = (np.array(rows_), ns)
    return out


MBR_MATS = {}
for name, (f, ns) in _read_meme(C["meme_path"]).items():
    f = f / f.sum(1, keepdims=True)
    pp = (f * ns + 0.25) / (ns + 1)
    S = pp * (pp.max(1) - pp.min(1))[:, None]
    S = np.vstack([S.T, np.zeros(len(S))]).T
    MBR_MATS[name] = (S, S[::-1][:, [3, 2, 1, 0, 4]], S[:, :4].min(1).sum(), S[:, :4].max(1).sum())
ENC = {"A": 0, "C": 1, "G": 2, "T": 3}
WIN = {}
for h in MWJ["hits"]:
    WIN.setdefault(h["region"], []).append(h)


def load_fits(path, keyed=True):
    # P(bound) per hit x layer (index (region, hit_id), or hit_id for the legacy per-region fits), AUC per TF x layer
    dec, auc, nl = [], {}, None
    for f in sorted(os.listdir(path)):
        if not f.endswith(".npz") or f.endswith(".part.npz"):
            continue
        z = np.load(f"{path}/{f}", allow_pickle=False)
        if "dec_hit" not in z or not len(z["hit_id"]):
            continue
        lids = [int(l) for l in (z["layers"] if "layers" in z else np.arange(z["dec_hit"].shape[1]))]
        idx = (pd.MultiIndex.from_arrays([z["hit_region"].astype(int), z["hit_id"].astype(int)], names=["region", "hit_id"])
               if keyed else z["hit_id"].astype(int))
        dec.append(pd.DataFrame(z["dec_hit"], index=idx, columns=lids))
        auc[f[:-4]] = pd.Series(np.asarray(z["auc"], float), index=lids)
        nl = int(z["n_layers"]) if "n_layers" in z else z["dec_hit"].shape[1] - 1
    return 1 / (1 + np.exp(-pd.concat(dec))), pd.DataFrame(auc).T, nl


def build(r, HS):
    # SNV x hit rows and motifbreakR pairs on the hits HS of region r (logic of the cross-model notebook, 3.3)
    S_ = f"{POOL}/regions/r{r}"
    hits = pd.read_csv(f"{S_}/hits.csv").set_index("hit_id")
    snv = pd.read_csv(f"{S_}/snvs.csv")
    q = pd.read_csv(f"{S_}/snv_hit_dpwm.csv")
    q = q[q["hit_id"].isin(HS)].sort_values(["snv", "hit_id"]).reset_index(drop=True)
    sn = q["snv"].values
    st = np.r_[0, np.flatnonzero(sn[1:] != sn[:-1]) + 1]
    hi = np.searchsorted(HS, q["hit_id"].values)
    n_snv = len(st)
    MOT = hits.loc[HS, "motif"].values
    base = {}
    for h in WIN[r]:
        for i, b in enumerate(h["window"]):
            base.setdefault(h["start"] - C["motif_offset"] + i, b)
    lo = min(base)
    seq = np.array([ENC.get(base.get(x, "N"), 4) for x in range(lo, max(base) + 1)])
    s_ = snv.loc[sn[st]]
    pos, ref, alt = s_["local"].values - lo, s_["ref"].map(ENC).values, s_["alt"].map(ENC).values
    assert (seq[pos] == ref).all(), "rebuilt sequence disagrees with the SNV reference bases"
    out = []
    for m in sorted(set(MOT)):
        if m not in MBR_MATS:
            continue
        S, Sr, smin, smax = MBR_MATS[m]
        w = len(S)
        n = len(seq) - w + 1
        idx = np.arange(n)[:, None] + np.arange(w)[None, :]
        f_fwd, f_rev = S[np.arange(w), seq[idx]].sum(1), Sr[np.arange(w), seq[idx]].sum(1)
        stw = pos[:, None] - np.arange(w)[None, :]
        k = np.broadcast_to(np.arange(w), stw.shape)
        ok = (stw >= 0) & (stw < n)
        stc = np.clip(stw, 0, n - 1)
        ref_f, ref_r = np.where(ok, f_fwd[stc], -np.inf), np.where(ok, f_rev[stc], -np.inf)
        alt_f = ref_f + np.where(ok, S[k, alt[:, None]] - S[k, ref[:, None]], 0)
        alt_r = ref_r + np.where(ok, Sr[k, alt[:, None]] - Sr[k, ref[:, None]], 0)
        sr, sa = np.maximum(ref_f, ref_r).max(1), np.maximum(alt_f, alt_r).max(1)
        pr, pa = (sr - smin) / (smax - smin), (sa - smin) / (smax - smin)
        diff = sa - sr
        keep = np.isfinite(sr) & ((pr > MBR_THRESH) | (pa > MBR_THRESH)) & (np.abs(diff) >= MBR_NEUT)
        out += [(i, m, diff[i]) for i in np.flatnonzero(keep)]
    t = pd.DataFrame(out, columns=["i", "motif", "alleleDiff"])
    t["j"] = np.arange(len(t))
    rows_ = pd.DataFrame({"i": np.repeat(np.arange(n_snv), np.diff(np.r_[st, len(sn)])), "motif": MOT[hi], "h": hi})
    link = t[["j", "i", "motif"]].merge(rows_, on=["i", "motif"])
    keep = np.sort(link["j"].unique())
    t = t.loc[keep].reset_index(drop=True)
    # every base of the region with its mean measured log2FC; SNVs outside the hits get prediction 0
    pos_all = np.arange(snv["local"].min(), snv["local"].max() + 1)
    meas = snv.groupby("local")["log2FC"].mean().reindex(pos_all).values
    pidx = snv["local"].values - pos_all[0]
    return dict(r=r, HS=HS, MOT=MOT, hits=hits, snv=snv, n_snv=n_snv, snv_id=sn[st],
                y=snv.loc[sn[st], "log2FC"].values, t=t, lj=np.searchsorted(keep, link["j"].values),
                lh=link["h"].values, DIFF=t["alleleDiff"].values, TI=t["i"].values, MOTP=t["motif"].values,
                pos_all=pos_all, meas=meas, tr_idx=pidx[sn[st]], cnt=np.bincount(pidx, minlength=len(pos_all)))


def predict(R, p_hit):
    # per SNV: the three rules, plus the P(bound) and motif of the CAV's choice and motifbreakR's |alleleDiff|
    P = np.full(len(R["t"]), -np.inf)
    np.maximum.at(P, R["lj"], p_hit[R["lh"]])
    TI, DIFF, n = R["TI"], R["DIFF"], R["n_snv"]

    def first(key):
        o = np.lexsort((-key, TI))
        return o[np.r_[True, TI[o][1:] != TI[o][:-1]]]

    fb, fc = first(np.abs(DIFF)), first(P + 1e-9 * np.abs(DIFF))
    out = {k: np.zeros(n) for k in ("motifbreakR", "CAV-chosen motifbreakR", "motifbreakR + CAV")}
    out["motifbreakR"][TI[fb]] = DIFF[fb]
    out["CAV-chosen motifbreakR"][TI[fc]] = DIFF[fc]
    out["motifbreakR + CAV"][TI[fc]] = DIFF[fc] * wfun(P[fc])
    out["P chosen"] = np.full(n, np.nan); out["P chosen"][TI[fc]] = P[fc]
    out["motif chosen"] = np.full(n, None, dtype=object); out["motif chosen"][TI[fc]] = R["MOTP"][fc]
    out["|alleleDiff| mbR"] = np.full(n, np.nan); out["|alleleDiff| mbR"][TI[fb]] = np.abs(DIFF[fb])
    return out


# ---------- fits ----------
PB, AUCM, NL, LAYER, PBR = {}, {}, {}, {}, {}
n_sites = len([m for m, s in MWJ["status"].items() if s == "built"])
for tag in MODELS:
    d = f"{POOL}/fits/{tag}"
    if not (os.path.isdir(d) and any(f.endswith(".npz") and not f.endswith(".part.npz") for f in os.listdir(d))):
        print(f"{tag}: no fits yet -- left out")
        continue
    PB[tag], AUCM[tag], NL[tag] = load_fits(d)
    LAYER[tag] = LAYER_FIX.get(tag, int(AUCM[tag].mean().idxmax()))
    PBR[tag] = {r: g.droplevel(0) for r, g in PB[tag][LAYER[tag]].groupby(level=0)}
    done = len(AUCM[tag])
    print(f"{tag}: {done} of {n_sites} TFs fitted{'' if done == n_sites else '  <-- INCOMPLETE: hits of the missing TFs are left out for every model'}; "
          f"layer {LAYER[tag]}/{NL[tag]}, mean held-out AUC " + "  ".join(f"L{l} {a:.3f}" for l, a in AUCM[tag].mean().items()))
MODELS = [t for t in MODELS if t in PB]
COL = {tg: f"C{i}" for i, tg in enumerate(MODELS)}

# ---------- regions: the hits every model has a direction for ----------
CTX, rows = {}, []
for r in sorted(INFO):
    sets = [set(PBR[t][r].index) if r in PBR[t] else set() for t in MODELS]
    HS = np.array(sorted(set.intersection(*sets))) if sets else np.array([])
    row = {"region": r, "cell": INFO[r]["cell"], "hits shared": len(HS)}
    if len(HS):
        R = build(r, HS)
        row.update({"SNVs in them": R["n_snv"], "SNVs with a motifbreakR call": R["t"]["i"].nunique(),
                    "TFs": len(set(R["MOT"])), "bases": len(R["pos_all"])})
        if R["t"]["i"].nunique() >= MIN_SNV:
            R["home"] = set(INFO[r]["cell"].split(",")) <= HOME
            CTX[r] = R
    row["used"] = r in CTX
    rows.append(row)
print()
print(pd.DataFrame(rows).set_index("region").to_string())
REGS = sorted(CTX)
GROUPS = {"all regions": REGS, "HepG2/K562": [r for r in REGS if CTX[r]["home"]],
          "other cells": [r for r in REGS if not CTX[r]["home"]]}
GROUPS = {g: rs for g, rs in GROUPS.items() if rs}
print("\n" + "\n".join(f"{g}: {len(rs)} regions {rs}" for g, rs in GROUPS.items()))

## 1.3 Check against the earlier per-region fits

Regions 0, 8 and 20 had their own NT-v2 500M fits before. The pooled fit uses the same sites minus the pairs whose
windows overlap one of the other MPRA regions, so P(bound) at L24 should agree closely (Spearman near 1), and
motifbreakR + CAV should give nearly the same Pearson as the 2026-10-20 write-up (0.404 / 0.625 / 0.579; motifbreakR
alone 0.177 / 0.279 / 0.464). motifbreakR alone uses no fit, so it should match whenever the same hits are used.

In [ ]:
def per_snv(pred, y):
    if np.std(pred) == 0:
        return np.array([np.nan] * 3)
    return np.array([stats.spearmanr(pred, y)[0], stats.spearmanr(np.abs(pred), np.abs(y))[0], stats.pearsonr(pred, y)[0]])


WANT = {0: (0.177, 0.404), 8: (0.279, 0.625), 20: (0.464, 0.579)}
OLD_SRC = {0: "fcav_pgc_hk_r0", 8: "fcav_pgc_hk", 20: "fcav_pgc_hk_r20"}
if "nt500m" in PB and 24 in PB["nt500m"].columns:
    rows = []
    for r, lg in LEGACY.items():
        p_old = f"{WORK_DIR}/{lg}/fits"
        if r not in PBR["nt500m"] or not os.path.isdir(p_old):
            continue
        old = load_fits(p_old, keyed=False)[0][24]
        old_h = pd.read_csv(f"{WORK_DIR}/{OLD_SRC[r]}/hits.csv")
        new_h = pd.read_csv(f"{POOL}/regions/r{r}/hits.csv")
        same_hits = (len(old_h) == len(new_h)) and (old_h[["motif", "start", "end", "strand"]].values == new_h[["motif", "start", "end", "strand"]].values).all()
        new = PB["nt500m"][24].xs(r, level="region")
        both = old.index.intersection(new.index)
        R0 = build(r, np.array(sorted(new.index)))
        pr = predict(R0, new.loc[R0["HS"]].values)
        rows.append({"region": r, "same hits as before": same_hits, "hits: before / now / both": f"{len(old)} / {len(new)} / {len(both)}",
                     "Spearman P(bound) before vs now": stats.spearmanr(old.loc[both], new.loc[both])[0] if same_hits else np.nan,
                     "max |ΔP|": np.abs(old.loc[both] - new.loc[both]).max() if same_hits else np.nan,
                     "motifbreakR Pearson": per_snv(pr["motifbreakR"], R0["y"])[2], "write-up": WANT[r][0],
                     "motifbreakR + CAV Pearson": per_snv(pr["motifbreakR + CAV"], R0["y"])[2], "write-up ": WANT[r][1]})
    print(pd.DataFrame(rows).set_index("region").to_string(float_format=lambda x: f"{x:.3f}"))
else:
    print("nt500m at L24 not available; check skipped")

---
# Section 2 — every region, and the pooled comparison

## 2.1 Scores per region and model

In [ ]:
def track(R, pred):
    # per-SNV prediction -> mean per base over the region (all SNVs at a base; 0 where nothing is predicted)
    s = np.bincount(R["tr_idx"], weights=pred, minlength=len(R["pos_all"]))
    return np.divide(s, R["cnt"], out=np.zeros_like(s), where=R["cnt"] > 0)


def smooth(a, w):
    # centred moving average ignoring NaN (= pandas rolling(w, center=True, min_periods=1).mean() for odd w)
    if w == 1:
        return a
    ok = np.isfinite(a)
    num = np.convolve(np.where(ok, a, 0.0), np.ones(w), "same")
    den = np.convolve(ok.astype(float), np.ones(w), "same")
    return np.divide(num, den, out=np.full(len(a), np.nan), where=den > 0)


def window_pearson(meas, pred, w):
    ok = np.isfinite(meas)
    m, p = smooth(meas, w)[ok], smooth(pred, w)[ok]
    return np.nan if np.std(p) == 0 else stats.pearsonr(p, m)[0]


def dilate(b, k):
    return b if k == 0 else np.convolve(b.astype(float), np.ones(2 * k + 1), "same") > 0


def effects(meas):
    ok = np.isfinite(meas)
    m = np.where(ok, meas, 0.0)
    thr = np.quantile(np.abs(m[ok]), 1 - EFFECT_TOP)
    return ok & (m <= -thr), ok & (m >= thr), ok


def agreement(meas, pred, k, pred_min=PRED_MIN, E=None):
    En, Ep, ok = E if E is not None else effects(meas)
    Pn, Pp = ok & (pred <= -pred_min), ok & (pred >= pred_min)
    n_pred, n_eff = (Pn | Pp).sum(), (En | Ep).sum()
    right = ((Pn & dilate(En, k)) | (Pp & dilate(Ep, k))).sum()
    found = ((En & dilate(Pn, k)) | (Ep & dilate(Pp, k))).sum()
    prec = right / n_pred if n_pred else np.nan
    rec = found / n_eff
    f1 = 2 * prec * rec / (prec + rec) if n_pred and prec + rec > 0 else 0.0
    return prec, rec, f1, int(n_pred)


METRICS = ["SNV Pearson", "SNV Spearman |.|", f"w{W_MAIN} Pearson", f"±{K_MAIN} F1"]


def metrics(R, pred):
    tr = track(R, pred)
    s = per_snv(pred, R["y"])
    return np.array([s[2], s[1], window_pearson(R["meas"], tr, W_MAIN), agreement(R["meas"], tr, K_MAIN, E=R["E"])[2]])


RULES = ["motifbreakR", "CAV-chosen motifbreakR", "motifbreakR + CAV"]
PRED, rows = {}, []
for r in REGS:
    R = CTX[r]
    R["E"] = effects(R["meas"])
    R["zy"] = (R["y"] - R["y"].mean()) / R["y"].std()
    for tag in MODELS:
        p = PBR[tag][r].loc[R["HS"]].values
        PRED[r, tag] = predict(R, p)
        for rule in RULES:
            if rule == "motifbreakR" and tag != MODELS[0]:
                continue                                   # the same for every model (shared hits): keep one copy
            rows.append({"region": r, "cell": INFO[r]["cell"], "home": R["home"], "model": "-" if rule == "motifbreakR" else tag,
                         "rule": rule, **dict(zip(METRICS, metrics(R, PRED[r, tag][rule]))),
                         "share of hits with P >= 0.75": np.nan if rule == "motifbreakR" else np.mean(p >= P_CONF)})
EV = pd.DataFrame(rows)
EV.to_csv(f"{OUT}/scores_by_region.csv", index=False)
wide = EV[EV.rule != "CAV-chosen motifbreakR"].pivot_table(index=["region", "cell"], columns="model", values=METRICS)
print(f"per region: '-' = motifbreakR, model columns = motifbreakR + CAV with that model\n")
print(wide.to_string(float_format=lambda x: f"{x:+.3f}"))

## 2.2 Pooled over regions, with nulls

Two summaries per model and group of regions:

* **mean over regions** of each score (every region counts the same), and in how many regions motifbreakR + CAV beats
  motifbreakR;
* **pooled SNV Pearson**: all SNVs of the group in one correlation, the measured log2FC standardised within its region
  (regions differ in scale), the prediction as is.

Nulls (1000 each), P(bound) shuffled and the CAV's choice redone: **N1** within each region across its hits; **N2**
within each TF across all its hits in all regions (keeps which TF a hit belongs to and asks whether the CAV's number
for this particular occurrence matters). p = share of shuffles at least as good as observed.

In [ ]:
def summary(preds):
    # preds: {region: per-SNV prediction} -> per group: ([mean over regions of each metric..., pooled SNV Pearson],
    # the per-region metrics)
    M = {r: metrics(CTX[r], preds[r]) for r in REGS}
    out = {}
    for g, rs in GROUPS.items():
        pooled = stats.pearsonr(np.concatenate([preds[r] for r in rs]), np.concatenate([CTX[r]["zy"] for r in rs]))[0]
        Mg = np.array([M[r] for r in rs])
        out[g] = (np.r_[np.nanmean(Mg, 0), pooled], Mg)
    return out


STATS = METRICS + ["pooled SNV Pearson"]
OFF = np.cumsum([0] + [len(CTX[r]["HS"]) for r in REGS])
MALL = np.concatenate([CTX[r]["MOT"] for r in REGS])
G2 = [g for g in (np.flatnonzero(MALL == m) for m in np.unique(MALL)) if len(g) > 1]
print(f"{len(MALL)} hits over {len(REGS)} regions, {len(np.unique(MALL))} TFs; {sum(len(g) for g in G2)} hits belong to a TF "
      f"with more than one hit (what N2 can shuffle), {len(G2)} such TFs")


def split(p):
    return {r: p[OFF[i]:OFF[i + 1]] for i, r in enumerate(REGS)}


def shuffle_n1(ps, rng):
    return {r: rng.permutation(p) for r, p in ps.items()}


def shuffle_n2(ps, rng):
    q = np.concatenate([ps[r] for r in REGS])
    for g in G2:
        q[g] = q[g][rng.permutation(len(g))]
    return split(q)


base_preds = {r: PRED[r, MODELS[0]]["motifbreakR"] for r in REGS}
BASE = summary(base_preds)
rng = np.random.default_rng(7)
rows, NULLS = [], {}
for tag in MODELS:
    ps = {r: PBR[tag][r].loc[CTX[r]["HS"]].values for r in REGS}
    obs = summary({r: PRED[r, tag]["motifbreakR + CAV"] for r in REGS})
    null = {g: {"N1": [], "N2": []} for g in GROUPS}
    for _ in range(N_PERM):
        for nm, fn in (("N1", shuffle_n1), ("N2", shuffle_n2)):
            pp = fn(ps, rng)
            sm = summary({r: predict(CTX[r], pp[r])["motifbreakR + CAV"] for r in REGS})
            for g in GROUPS:
                null[g][nm].append(sm[g][0])
    NULLS[tag] = null
    for g, rs in GROUPS.items():
        (bm, bM), (om, oM) = BASE[g], obs[g]
        for j, s in enumerate(STATS):
            n1, n2 = np.array(null[g]["N1"])[:, j], np.array(null[g]["N2"])[:, j]
            rows.append({"group": g, "regions": len(rs), "model": f"{tag} L{LAYER[tag]}", "score": s,
                         "motifbreakR": bm[j], "motifbreakR + CAV": om[j], "gain": om[j] - bm[j],
                         "regions won": f"{int(np.sum(oM[:, j] > bM[:, j]))}/{len(rs)}" if j < len(METRICS) else "",
                         "p N1": np.mean(n1 >= om[j]), "p N2": np.mean(n2 >= om[j])})
    print(f"{tag} done", flush=True)
POOLED = pd.DataFrame(rows)
POOLED.to_csv(f"{OUT}/pooled.csv", index=False)
for s in STATS:
    print(f"\n=== {s}" + ("  (mean over regions)" if s in METRICS else "  (all SNVs, log2FC standardised within region)"))
    print(POOLED[POOLED.score == s].drop(columns="score").set_index(["group", "model"]).to_string(float_format=lambda x: f"{x:+.3f}"))

## 2.3 The gain in every region

One dot per (region, model): motifbreakR + CAV minus motifbreakR. Regions ordered by their motifbreakR score; the
grey band marks the HepG2/K562 regions.

In [ ]:
OURS = "motifbreakR + CAV"
SHOW = ["SNV Pearson", f"±{K_MAIN} F1"]
b = EV[EV.rule == "motifbreakR"].set_index("region")
fig, ax = plt.subplots(len(SHOW), 1, figsize=(max(8, 0.55 * len(REGS) + 2), 3.4 * len(SHOW)), sharex=True)
for a, s in zip(np.atleast_1d(ax), SHOW):
    order = b[s].sort_values().index.tolist()
    xs = {r: i for i, r in enumerate(order)}
    for i, r in enumerate(order):
        if CTX[r]["home"]:
            a.axvspan(i - .5, i + .5, color="#ececea", zorder=0)
    for k, tag in enumerate(MODELS):
        e = EV[(EV.rule == OURS) & (EV.model == tag)].set_index("region")
        a.scatter([xs[r] + (k - len(MODELS) / 2) * .1 for r in order], [e.loc[r, s] - b.loc[r, s] for r in order],
                  s=22, c=COL[tag], label=tag, zorder=3)
    a.axhline(0, c=INK, lw=.8)
    a.set_ylabel(f"gain in {s}")
    a.set_xticks(range(len(order)), [f"{r}\n{INFO[r]['cell'][:9]}" for r in order], fontsize=7)
np.atleast_1d(ax)[0].legend(frameon=False, fontsize=7, ncol=len(MODELS))
np.atleast_1d(ax)[0].set_title("motifbreakR + CAV minus motifbreakR, per region (grey = HepG2/K562; left to right: motifbreakR's own score rising)",
                               loc="left", fontsize=10)
plt.tight_layout(); plt.savefig(f"{OUT}/gain_by_region.png", dpi=150, bbox_inches="tight"); plt.show()

---
# Section 3 — when does the CAV help?

Two region-level questions from the meeting, now with one point per region instead of three:

* **Number of peaks.** A region with one or two strong peaks is decided by one motif choice. Peaks = runs of ≥ 3 bp
  where the 5 bp-smoothed |log2FC| is at least half the region's maximum.
* **The CAV's own confidence** (no MPRA used): the share of the region's hits with P(bound) ≥ 0.75, and the mean
  held-out AUC of its hits' TFs. On the regions where the CAV is most confident, does it beat motifbreakR?

Spearman over regions; each model separately.

In [ ]:
def n_peaks(meas, w=5, frac=PEAK_FRAC, min_len=3):
    s = smooth(np.abs(meas), w)
    s = np.where(np.isfinite(s), s, 0.0)
    on = np.r_[False, s >= frac * s.max(), False].astype(int)
    d = np.diff(on)
    starts, ends = np.flatnonzero(d == 1), np.flatnonzero(d == -1)
    return int(np.sum(ends - starts >= min_len))


rows = []
for r in REGS:
    R = CTX[r]
    for tag in MODELS:
        p = PBR[tag][r].loc[R["HS"]].values
        e = EV[(EV.region == r) & (EV.rule == OURS) & (EV.model == tag)].iloc[0]
        bb = EV[(EV.region == r) & (EV.rule == "motifbreakR")].iloc[0]
        rows.append({"region": r, "home": R["home"], "model": tag, "peaks": n_peaks(R["meas"]),
                     "SNVs with a call": R["t"]["i"].nunique(), "share P >= 0.75": np.mean(p >= P_CONF),
                     "mean AUC of the hits' TFs": np.nanmean(AUCM[tag][LAYER[tag]].reindex(R["MOT"]).values),
                     **{f"gain {s}": e[s] - bb[s] for s in METRICS}})
RG = pd.DataFrame(rows)
RG.to_csv(f"{OUT}/region_features.csv", index=False)
print(RG[RG.model == MODELS[0]].set_index("region")[["home", "peaks", "SNVs with a call"]].T.to_string())

XS = ["peaks", "SNVs with a call", "share P >= 0.75", "mean AUC of the hits' TFs"]
GS = [f"gain {s}" for s in ("SNV Pearson", f"±{K_MAIN} F1")]
out = []
for tag in MODELS:
    d = RG[RG.model == tag]
    for x in XS:
        out.append({"model": tag, "x": x, **{f"Spearman with {g}": stats.spearmanr(d[x], d[g], nan_policy="omit")[0] for g in GS}})
print("\nSpearman over regions (positive = more gain where x is larger):")
print(pd.DataFrame(out).set_index(["x", "model"]).to_string(float_format=lambda x: f"{x:+.2f}"))

# the regions where the CAV is most confident vs the rest
out = []
for tag in MODELS:
    d = RG[RG.model == tag]
    hi_ = d["share P >= 0.75"] >= d["share P >= 0.75"].median()
    for name, sel in (("more confident half", hi_), ("less confident half", ~hi_)):
        out.append({"model": tag, "regions": name, "n": int(sel.sum()),
                    **{g + " (mean)": d.loc[sel, g].mean() for g in GS},
                    **{g.replace("gain", "won") : f"{int((d.loc[sel, g] > 0).sum())}/{int(sel.sum())}" for g in GS}})
print("\nsplit at the median share of hits with P >= 0.75:")
print(pd.DataFrame(out).set_index(["model", "regions"]).to_string(float_format=lambda x: f"{x:+.3f}"))

fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for a, x in zip(ax, ["peaks", "share P >= 0.75"]):
    for tag in MODELS:
        d = RG[RG.model == tag]
        a.scatter(d[x] + (np.random.default_rng(0).uniform(-.12, .12, len(d)) if x == "peaks" else 0), d[GS[0]],
                  s=24, c=COL[tag], label=tag, marker="o")
    a.axhline(0, c=INK, lw=.8)
    a.set_xlabel(x)
ax[0].set_ylabel(GS[0]); ax[0].legend(frameon=False, fontsize=7)
ax[0].set_title("one dot per (region, model)", loc="left", fontsize=10)
plt.tight_layout(); plt.savefig(f"{OUT}/gain_vs_peaks_confidence.png", dpi=150, bbox_inches="tight"); plt.show()

---
# Section 4 — per-occurrence calibration over every region

Part C of the tolerant-metric notebook, unchanged in definition: one row per motif occurrence the CAV picks; its
P(bound) against the mean measured |log2FC| percentile (within region) of the SNVs where it was picked; and the same
after removing what motifbreakR's own score (its largest |alleleDiff|) explains. Before: 66–81 occurrences from 2–3
regions, and N2 could not shuffle anything because most TFs occurred once. Now N2 shuffles within TF across regions.

In [ ]:
def pct_within(v):
    return 100 * stats.rankdata(v) / len(v)


def chosen_hits(R, p_hit):
    # per SNV, the hit (index into HS) the CAV picks; -1 where it makes no call. Same choice as predict().
    lj, lh, TI, DIFF = R["lj"], R["lh"], R["TI"], R["DIFF"]
    ph = p_hit[lh]
    o = np.lexsort((-ph, lj))
    first = o[np.r_[True, lj[o][1:] != lj[o][:-1]]]
    H = np.full(len(R["t"]), -1)
    H[lj[first]] = lh[first]
    P = np.full(len(R["t"]), -np.inf)
    P[lj[first]] = ph[first]
    o = np.lexsort((-(P + 1e-9 * np.abs(DIFF)), TI))
    fc = o[np.r_[True, TI[o][1:] != TI[o][:-1]]]
    hit = np.full(R["n_snv"], -1)
    hit[TI[fc]] = H[fc]
    return hit


def partial_spearman(x, y, z):
    rx, ry, rz = stats.rankdata(x), stats.rankdata(y), stats.rankdata(z)
    res = lambda a: a - np.polyval(np.polyfit(rz, a, 1), rz)
    return stats.pearsonr(res(rx), res(ry))[0]


BASEC = {}
for r in REGS:
    R = CTX[r]
    has = np.zeros(R["n_snv"], bool); has[R["TI"]] = True
    ypct = np.full(R["n_snv"], np.nan); ypct[has] = pct_within(np.abs(R["y"][has]))
    ad = np.abs(predict(R, np.zeros(len(R["HS"])))["motifbreakR"])
    BASEC[r] = (has, ypct, ad)


def hit_rows(r, p):
    R = CTX[r]
    has, ypct, ad = BASEC[r]
    hit = chosen_hits(R, p)
    y_, a_ = ypct[has], ad[has]
    h_ = hit[has]
    u, inv, cnt = np.unique(h_, return_inverse=True, return_counts=True)
    return pd.DataFrame({"region": r, "hit": u, "motif": R["MOT"][u], "P": p[u], "n SNVs": cnt,
                         "y": np.bincount(inv, weights=y_) / cnt, "ad": np.bincount(inv, weights=a_) / cnt})


def two(g):
    return np.array([stats.spearmanr(g["P"], g["y"])[0], partial_spearman(g["P"], g["y"], g["ad"])])


rng = np.random.default_rng(11)
rows, OCC = [], {}
for tag in MODELS:
    ps = {r: PBR[tag][r].loc[CTX[r]["HS"]].values for r in REGS}
    for r, p in ps.items():                       # the hit picked here must be the motif predict() picked
        h, m = chosen_hits(CTX[r], p), PRED[r, tag]["motif chosen"]
        call = np.array([x is not None for x in m])
        assert ((h >= 0) == call).all() and (CTX[r]["MOT"][h[call]] == m[call]).all(), f"{tag} r{r}: choice mismatch"
    g = pd.concat([hit_rows(r, p) for r, p in ps.items()], ignore_index=True)
    OCC[tag] = g
    null = {gr: {"N1": [], "N2": []} for gr in GROUPS}
    for _ in range(N_PERM):
        for nm, fn in (("N1", shuffle_n1), ("N2", shuffle_n2)):
            gg = pd.concat([hit_rows(r, p) for r, p in fn(ps, rng).items()], ignore_index=True)
            for gr, rs in GROUPS.items():
                null[gr][nm].append(two(gg[gg.region.isin(rs)]))
    for gr, rs in GROUPS.items():
        d = g[g.region.isin(rs)]
        obs = two(d)
        n1, n2 = np.array(null[gr]["N1"]), np.array(null[gr]["N2"])
        mult = d.groupby("motif")["region"].transform("nunique") > 1
        rows.append({"group": gr, "model": f"{tag} L{LAYER[tag]}", "occurrences": len(d), "TFs": d["motif"].nunique(),
                     "occurrences of a TF seen in >1 region": int(mult.sum()),
                     "Spearman(P, effect)": obs[0], "p N1": np.mean(n1[:, 0] >= obs[0]), "p N2": np.mean(n2[:, 0] >= obs[0]),
                     "after removing |alleleDiff|": obs[1], "p N1 ": np.mean(n1[:, 1] >= obs[1]), "p N2 ": np.mean(n2[:, 1] >= obs[1]),
                     "Spearman(|alleleDiff|, effect)": stats.spearmanr(d["ad"], d["y"])[0]})
    print(f"{tag} done", flush=True)
HC = pd.DataFrame(rows)
HC.to_csv(f"{OUT}/calibration_per_occurrence.csv", index=False)
pd.concat(OCC, names=["model"]).to_csv(f"{OUT}/calibration_occurrences.csv")
print(f"\none row per motif occurrence the CAV picks; effect = mean measured |log2FC| percentile of its SNVs; "
      f"{N_PERM} shuffles of P (N1 within region, N2 within TF across regions)\n")
print(HC.set_index(["group", "model"]).to_string(float_format=lambda x: f"{x:+.3f}"))

## 4.2 The same as a picture

One dot per occurrence, all regions; x = P(bound) of the CAV's pick, y = its effect percentile. Lines: mean y in
fifths of P. Left: HepG2/K562 regions; right: the others.

In [ ]:
gl = [g for g in ("HepG2/K562", "other cells") if g in GROUPS]
fig, ax = plt.subplots(len(gl), len(MODELS), figsize=(3.3 * len(MODELS), 3.2 * len(gl)), sharex=True, sharey=True, squeeze=False)
for i, gr in enumerate(gl):
    for j, tag in enumerate(MODELS):
        a = ax[i, j]
        d = OCC[tag][OCC[tag].region.isin(GROUPS[gr])]
        a.scatter(d["P"], d["y"], s=8, c="#bdbdb8", lw=0)
        if len(d) >= 25:
            f = pd.qcut(d["P"].rank(method="first"), 5, labels=False)
            s = d.groupby(f).agg(P=("P", "mean"), y=("y", "mean"))
            a.plot(s["P"], s["y"], c=COL[tag], lw=2.2, marker="o", ms=4)
        a.axhline(50, c=INK, lw=.6, ls=":")
        a.set_title(f"{tag}, {gr} ({len(d)})", loc="left", fontsize=9)
        if i == len(gl) - 1:
            a.set_xlabel("P(bound) of the CAV's pick")
    ax[i, 0].set_ylabel("measured |log2FC| pct")
plt.tight_layout(); plt.savefig(f"{OUT}/calibration_per_occurrence.png", dpi=150, bbox_inches="tight"); plt.show()

## End

Outputs in `mpra_eval_pool/`: `scores_by_region.csv`, `pooled.csv`, `region_features.csv`,
`calibration_per_occurrence.csv`, `calibration_occurrences.csv`, and the three figures.